# BTP vs Bund: Asset Swap Spread Basis Trade

The classic EUR rates RV trade: comparing Italian BTPs against German Bunds via asset swap spreads.

**What we build:**
1. EUR risk-free curve (ESTR/OIS from swap rates)
2. Bund curve (from German government bond prices)
3. BTP prices at realistic spreads over Bunds
4. Asset swap spreads for both — par/par and proceeds conventions
5. BTP-Bund ASW basis as a trading signal
6. Z-spread vs ASW comparison

Market data modelled on **Jul 2024** — realistic EUR curve and sovereign spread levels.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import numpy as np
import pandas as pd
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.bond import FixedRateBond
from pricebook.bootstrap import bootstrap
from pricebook.bond_desk import fit_curve_from_bonds
from pricebook.bond_trading_desk import bond_risk_metrics, bond_carry_roll
from pricebook.par_asset_swap import ParAssetSwap, ProceedsAssetSwap, asw_vs_zspread
from pricebook.asset_swap_desk import asw_risk_metrics, asw_carry_decomposition
from pricebook.day_count import DayCountConvention
from pricebook.schedule import Frequency
from pricebook.viz import configure_theme
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

REF = date(2024, 7, 15)
print(f"Valuation date: {REF}")

## 1. EUR Risk-Free Curve (ESTR/OIS)

In [ ]:
# ESTR/OIS curve from EUR swap market (Jul 2024 levels)
# ECB deposit rate was 3.75% — short end near that, long end lower
eur_deposits = [
    (REF + relativedelta(months=1), 0.0370),
    (REF + relativedelta(months=3), 0.0368),
    (REF + relativedelta(months=6), 0.0355),
]
eur_swaps = [
    (REF + relativedelta(years=1),  0.0340),
    (REF + relativedelta(years=2),  0.0310),
    (REF + relativedelta(years=3),  0.0295),
    (REF + relativedelta(years=5),  0.0280),
    (REF + relativedelta(years=7),  0.0275),
    (REF + relativedelta(years=10), 0.0275),
    (REF + relativedelta(years=15), 0.0280),
    (REF + relativedelta(years=20), 0.0275),
    (REF + relativedelta(years=30), 0.0260),
]
eur_curve = bootstrap(REF, eur_deposits, eur_swaps)

print("ESTR/OIS Curve (EUR risk-free)")
print(f"{'Tenor':>6}  {'Zero Rate':>9}")
print(f"{'─'*6}  {'─'*9}")
for t in [0.25, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30]:
    d = REF + relativedelta(months=int(t*12)) if t < 1 else REF + relativedelta(years=int(t))
    print(f"{t:>5}Y  {eur_curve.zero_rate(d)*100:>8.2f}%")

## 2. German Bunds — the EUR benchmark

Bunds: annual coupon, ACT/ACT, T+2 settlement, decimal quoting.

In [ ]:
def bund(issue: date, maturity: date, coupon: float) -> FixedRateBond:
    """German Bund: annual coupon, ACT/ACT, T+2."""
    return FixedRateBond(
        issue_date=issue, maturity=maturity, coupon_rate=coupon,
        frequency=Frequency.ANNUAL,
        day_count=DayCountConvention.ACT_ACT_ICMA,
        settlement_days=2,
    )

def btp(issue: date, maturity: date, coupon: float) -> FixedRateBond:
    """Italian BTP: semi-annual coupon, ACT/ACT, T+2."""
    return FixedRateBond(
        issue_date=issue, maturity=maturity, coupon_rate=coupon,
        frequency=Frequency.SEMI_ANNUAL,
        day_count=DayCountConvention.ACT_ACT_ICMA,
        settlement_days=2,
    )

# Bund universe — on-the-run benchmarks (Jul 2024 levels)
# Bunds traded near ESTR with small negative ASW (Bunds rich to swaps)
bund_bonds = [
    ("2Y",  0.0275, date(2024, 4, 10), date(2026, 4, 10),  99.750),
    ("5Y",  0.0250, date(2024, 1, 10), date(2029, 1, 10),  99.125),
    ("10Y", 0.0275, date(2024, 1, 10), date(2034, 1, 10),  99.500),
    ("30Y", 0.0250, date(2024, 1, 10), date(2054, 1, 10),  97.000),
]

bund_pairs = []
for label, coupon, issue, mat, px in bund_bonds:
    b = bund(issue, mat, coupon)
    bund_pairs.append((b, px))

# Bund-fitted curve
bund_curve, bund_fitted = fit_curve_from_bonds(REF, bund_pairs)

print("Bund Universe")
print(f"{'Tenor':>5}  {'Coupon':>7}  {'Price':>8}  {'FitPx':>8}  {'Zero':>7}")
print(f"{'─'*5}  {'─'*7}  {'─'*8}  {'─'*8}  {'─'*7}")
for (label, cpn, issue, mat, px), fb in zip(bund_bonds, bund_fitted):
    zero = bund_curve.zero_rate(mat)
    print(f"{label:>5}  {cpn*100:>6.2f}%  {px:>8.3f}  {fb.fitted_price:>8.3f}  {zero*100:>6.2f}%")

## 3. Italian BTPs — sovereign credit spread over Bunds

BTPs: semi-annual coupon, ACT/ACT, T+2. Trade at a spread over Bunds reflecting Italian sovereign credit risk.

Jul 2024: BTP-Bund 10Y spread ≈ 130-140bp.

In [ ]:
# BTP universe — realistic spread levels (Jul 2024)
# Higher coupons, lower prices reflecting Italian credit spread
btp_bonds = [
    ("3Y",  0.0350, date(2024, 3, 1),  date(2027, 3, 1),   99.250),
    ("5Y",  0.0375, date(2024, 1, 15), date(2029, 1, 15),   98.500),
    ("7Y",  0.0400, date(2024, 3, 1),  date(2031, 3, 1),    97.750),
    ("10Y", 0.0425, date(2024, 1, 15), date(2034, 1, 15),  100.250),
    ("15Y", 0.0450, date(2024, 3, 1),  date(2039, 3, 1),    98.000),
    ("30Y", 0.0475, date(2024, 1, 15), date(2054, 1, 15),   96.500),
]

btp_objects = []
for label, coupon, issue, mat, px in btp_bonds:
    b = btp(issue, mat, coupon)
    btp_objects.append((b, px, label))

# Compute YTM for each BTP
print("BTP Universe")
print(f"{'Tenor':>5}  {'Coupon':>7}  {'Price':>8}  {'YTM':>7}")
print(f"{'─'*5}  {'─'*7}  {'─'*8}  {'─'*7}")
for (label, cpn, issue, mat, px) in btp_bonds:
    b = btp(issue, mat, cpn)
    settle = b.settlement_date(REF)
    ai = b.accrued_interest(settle)
    ytm = b.yield_to_maturity(px + ai, settle)
    print(f"{label:>5}  {cpn*100:>6.2f}%  {px:>8.3f}  {ytm*100:>6.2f}%")

## 4. Asset Swap Spreads — Bund vs BTP

The ASW spread strips out the rate component and isolates credit/liquidity.

- **Bund ASW** should be small or negative (Bunds are "special" — rich to swaps)
- **BTP ASW** should be large and positive (Italian credit risk)
- **BTP-Bund ASW basis** = the sovereign spread in swap-adjusted terms

In [ ]:
# Bund ASW spreads (against EUR OIS curve)
print("═" * 80)
print("BUND Asset Swap Spreads (vs ESTR/OIS)")
print(f"{'Tenor':>5}  {'Price':>8}  {'Par ASW':>9}  {'Proceeds':>9}  {'Z-spread':>9}")
print(f"{'─'*5}  {'─'*8}  {'─'*9}  {'─'*9}  {'─'*9}")

bund_asw_results = []
for (label, cpn, issue, mat, px), (bond_obj, _) in zip(bund_bonds, bund_pairs):
    settle = bond_obj.settlement_date(REF)
    
    par = ParAssetSwap(bond_obj, settle, px)
    par_r = par.price(eur_curve)
    
    ai = bond_obj.accrued_interest(settle)
    proceeds = ProceedsAssetSwap(bond_obj, settle, px + ai)
    proc_r = proceeds.price(eur_curve)
    
    comparison = asw_vs_zspread(bond_obj, px, eur_curve, settle)
    
    bund_asw_results.append((label, par_r.asw_spread, proc_r.asw_spread, comparison.z_spread))
    
    print(f"{label:>5}  {px:>8.3f}  {par_r.asw_spread*1e4:>+8.1f}bp  "
          f"{proc_r.asw_spread*1e4:>+8.1f}bp  {comparison.z_spread*1e4:>+8.1f}bp")

# BTP ASW spreads
print(f"\n{'═' * 80}")
print("BTP Asset Swap Spreads (vs ESTR/OIS)")
print(f"{'Tenor':>5}  {'Price':>8}  {'Par ASW':>9}  {'Proceeds':>9}  {'Z-spread':>9}")
print(f"{'─'*5}  {'─'*8}  {'─'*9}  {'─'*9}  {'─'*9}")

btp_asw_results = []
for (bond_obj, px, label) in btp_objects:
    settle = bond_obj.settlement_date(REF)
    
    par = ParAssetSwap(bond_obj, settle, px)
    par_r = par.price(eur_curve)
    
    ai = bond_obj.accrued_interest(settle)
    proceeds = ProceedsAssetSwap(bond_obj, settle, px + ai)
    proc_r = proceeds.price(eur_curve)
    
    comparison = asw_vs_zspread(bond_obj, px, eur_curve, settle)
    
    btp_asw_results.append((label, par_r.asw_spread, proc_r.asw_spread, comparison.z_spread))
    
    print(f"{label:>5}  {px:>8.3f}  {par_r.asw_spread*1e4:>+8.1f}bp  "
          f"{proc_r.asw_spread*1e4:>+8.1f}bp  {comparison.z_spread*1e4:>+8.1f}bp")

## 5. BTP-Bund ASW Basis — the trading signal

In [ ]:
# Match tenors where both Bund and BTP exist
bund_dict = {label: (par, proc, zs) for label, par, proc, zs in bund_asw_results}
btp_dict = {label: (par, proc, zs) for label, par, proc, zs in btp_asw_results}

common_tenors = sorted(set(bund_dict.keys()) & set(btp_dict.keys()),
                       key=lambda x: int(x.replace('Y', '')))

print("BTP-Bund ASW Basis (par/par convention)")
print(f"{'Tenor':>5}  {'Bund ASW':>9}  {'BTP ASW':>9}  {'Basis':>9}  {'Z-spread basis':>14}")
print(f"{'─'*5}  {'─'*9}  {'─'*9}  {'─'*9}  {'─'*14}")

basis_tenors = []
basis_values = []

for label in common_tenors:
    bund_par = bund_dict[label][0]
    btp_par = btp_dict[label][0]
    bund_zs = bund_dict[label][2]
    btp_zs = btp_dict[label][2]
    
    basis = (btp_par - bund_par) * 1e4
    zs_basis = (btp_zs - bund_zs) * 1e4
    
    basis_tenors.append(label)
    basis_values.append(basis)
    
    print(f"{label:>5}  {bund_par*1e4:>+8.1f}bp  {btp_par*1e4:>+8.1f}bp  "
          f"{basis:>+8.1f}bp  {zs_basis:>+13.1f}bp")

# Also show tenors only in BTP
btp_only = sorted(set(btp_dict.keys()) - set(bund_dict.keys()),
                  key=lambda x: int(x.replace('Y', '')))
if btp_only:
    print(f"\nBTP-only tenors (no matching Bund):")
    for label in btp_only:
        btp_par = btp_dict[label][0]
        print(f"{label:>5}  {'—':>9}  {btp_par*1e4:>+8.1f}bp")

In [ ]:
# Plot: ASW term structure + basis
bund_tenors_num = [int(l.replace('Y','')) for l in [r[0] for r in bund_asw_results]]
bund_par_bp = [r[1]*1e4 for r in bund_asw_results]

btp_tenors_num = [int(l.replace('Y','')) for l in [r[0] for r in btp_asw_results]]
btp_par_bp = [r[1]*1e4 for r in btp_asw_results]

with apply_theme():
    fig, (ax1, ax2) = create_figure(2)

    # ASW term structure
    ax1.plot(bund_tenors_num, bund_par_bp, 'o-', linewidth=2, label="Bund ASW")
    ax1.plot(btp_tenors_num, btp_par_bp, 's-', linewidth=2, label="BTP ASW")
    ax1.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax1.set_xlabel("Tenor (years)")
    ax1.set_ylabel("Par ASW Spread (bp)")
    ax1.set_title("Asset Swap Spread Term Structure")
    ax1.legend(fontsize=9)

    # BTP-Bund basis
    tenor_nums = [int(l.replace('Y','')) for l in basis_tenors]
    ax2.bar(basis_tenors, basis_values, alpha=0.8, color='tab:red')
    ax2.set_xlabel("Tenor")
    ax2.set_ylabel("BTP - Bund ASW (bp)")
    ax2.set_title("BTP-Bund ASW Basis (sovereign spread)")

    fig.tight_layout()

## 6. Risk & Carry comparison

In [ ]:
# ECB repo rate (Jul 2024 deposit facility = 3.75%)
ecb_repo = 0.0375

print("ASW Risk Metrics & Carry (10M notional, 30-day horizon)")
print(f"\n{'Bond':>12}  {'ASW (bp)':>9}  {'ASW01':>8}  {'DV01':>8}  {'Carry/day':>10}")
print(f"{'─'*12}  {'─'*9}  {'─'*8}  {'─'*8}  {'─'*10}")

# Bunds
for (label, cpn, issue, mat, px), (bond_obj, _) in zip(bund_bonds, bund_pairs):
    settle = bond_obj.settlement_date(REF)
    asw_instr = ParAssetSwap(bond_obj, settle, px)
    rm = asw_risk_metrics(asw_instr, eur_curve)
    carry = asw_carry_decomposition(asw_instr, eur_curve, repo_rate=ecb_repo, horizon_days=1)
    print(f"{'Bund '+label:>12}  {rm.asw_spread*1e4:>+8.1f}bp  {rm.asw01:>8.0f}  "
          f"{rm.dv01:>8.0f}  {carry.net_carry:>+10.0f}")

print()

# BTPs
for (bond_obj, px, label) in btp_objects:
    settle = bond_obj.settlement_date(REF)
    asw_instr = ParAssetSwap(bond_obj, settle, px)
    rm = asw_risk_metrics(asw_instr, eur_curve)
    carry = asw_carry_decomposition(asw_instr, eur_curve, repo_rate=ecb_repo, horizon_days=1)
    print(f"{'BTP '+label:>12}  {rm.asw_spread*1e4:>+8.1f}bp  {rm.asw01:>8.0f}  "
          f"{rm.dv01:>8.0f}  {carry.net_carry:>+10.0f}")

## 7. Z-spread vs ASW: where do they diverge?

In [ ]:
# Full comparison for all BTPs
print("Z-spread vs ASW comparison (BTPs)")
print(f"{'Tenor':>5}  {'Z-spread':>9}  {'Par ASW':>9}  {'Proc ASW':>9}  {'Par-Z':>7}  {'Proc-Z':>7}")
print(f"{'─'*5}  {'─'*9}  {'─'*9}  {'─'*9}  {'─'*7}  {'─'*7}")

zs_list, par_list, proc_list, tenor_list = [], [], [], []

for (bond_obj, px, label) in btp_objects:
    settle = bond_obj.settlement_date(REF)
    comp = asw_vs_zspread(bond_obj, px, eur_curve, settle)
    
    zs_list.append(comp.z_spread * 1e4)
    par_list.append(comp.par_asw_spread * 1e4)
    proc_list.append(comp.proceeds_asw_spread * 1e4)
    tenor_list.append(label)
    
    print(f"{label:>5}  {comp.z_spread*1e4:>+8.1f}bp  {comp.par_asw_spread*1e4:>+8.1f}bp  "
          f"{comp.proceeds_asw_spread*1e4:>+8.1f}bp  {comp.par_asw_basis:>+6.1f}  {comp.proceeds_asw_basis:>+6.1f}")

print("\nNote: Par ASW and Z-spread diverge for off-par bonds.")
print("Proceeds ASW stays closer to Z-spread (no upfront exchange).")

In [ ]:
# Plot Z-spread vs ASW
with apply_theme():
    fig, ax = create_figure(1)

    x = np.arange(len(tenor_list))
    width = 0.25
    ax.bar(x - width, zs_list, width, label='Z-spread', alpha=0.8)
    ax.bar(x, par_list, width, label='Par ASW', alpha=0.8)
    ax.bar(x + width, proc_list, width, label='Proceeds ASW', alpha=0.8)

    ax.set_xlabel('Tenor')
    ax.set_ylabel('Spread (bp)')
    ax.set_title('BTP: Z-spread vs Par ASW vs Proceeds ASW')
    ax.set_xticks(x)
    ax.set_xticklabels(tenor_list)
    ax.legend(fontsize=9)

    fig.tight_layout()

## Summary

| Concept | What it measures | Convention |
|---------|-----------------|------------|
| **ASW spread (par)** | Credit/liquidity vs OIS, buyer pays par | Standard for EUR govt bonds |
| **ASW spread (proceeds)** | Same, but buyer pays dirty price (no upfront) | Better for off-par bonds |
| **Z-spread** | Constant spread to OIS that reprices the bond | Model-free, no swap leg |
| **BTP-Bund ASW basis** | Italian sovereign spread in ASW terms | The trading signal |

**Key insights:**
- Bunds trade **tight to OIS** (near zero or slightly negative ASW) — they are the benchmark
- BTPs trade at a **positive ASW** reflecting Italian credit risk
- The **basis** (BTP ASW - Bund ASW) is the sovereign spread in swap-adjusted terms
- Par ASW and Z-spread **diverge for off-par bonds** — proceeds ASW tracks Z-spread more closely
- A BTP-Bund basis **tightener** = long BTP ASW + short Bund ASW (bet on Italy improving)
- A BTP-Bund basis **widener** = short BTP ASW + long Bund ASW (bet on Italy deteriorating)